# Step 8: Train Model 2 with a chemistry-pretrained base checkpoint (Colab GPU)

Same task/data as `06_train_conditions_model.ipynb` (predict solvent/catalyst/temperature/yield
from product+reactants), but swaps `t5-small` (a plain, non-chemical checkpoint) for
`sagawa/ReactionT5v2-retrosynthesis` (the same ORD-chemistry-pretrained checkpoint used for
Model 1) as the base for fine-tuning.

**Why:** Model 2's weakest fields (RESULTS.md) are the ones that need real chemistry
understanding (solvent 24.4%/catalyst 14.4% top-1) -- plausibly because `t5-small` starts
with zero chemistry knowledge (pretrained on generic web text, C4), unlike Model 1's
ReactionT5 base (43.7% zero-shot on ORD retrosynthesis before any fine-tuning at all).
This tests whether a chemistry-pretrained starting point helps Model 2 the same way it
helped Model 1.

**Same data, unchanged, for a valid comparison against the existing RESULTS.md numbers:**
`data/v2_ord_train/conditions_{train,val,test}.jsonl` (41,139 / 2,285 / 2,285, from the 60k
ORD pool). Do NOT switch to the bigger 150k/300k pools' conditions data for this -- checked,
and ~90% of the existing conditions_test/val split leaks into those bigger pools'
conditions_train (`build_train_data_ord.py` only excludes Model 1's `eval_targets.json` from
the pool, not Model 2's own test/val split, since that split is carved out *after* pool
sampling). Scaling Model 2's data is a legitimate follow-up, but needs that leak fixed
first -- out of scope here.

**Learning rate lowered to 5e-5** (from the script's default 5e-4, tuned for plain
non-chemical `t5-small`): mirrors exactly what fixed Model 1's v1-\>v2 regression (a
chemistry-pretrained checkpoint fine-tuned with a too-high, non-chemical-model learning
rate degraded instead of improved). `load_best_model_at_end` (by `eval_loss`) is already on,
so an overly aggressive schedule would just get discarded rather than silently kept -- but
starting from the value that's actually proven for this checkpoint family is safer than
relying on that safety net alone.

**No measured timing baseline for this specific architecture+task combination.** Rough
estimate from Model 1's Kaggle DDP throughput (0.972 steps/s on 2xT4, scaled down for 1xT4
here): ~5,100 steps (41,139 examples / batch 32 x 4 epochs) at roughly half that per-GPU
rate ~= 2-2.5h -- should fit in a single ~3h Colab session, but watch the first `train.log`
lines after ~10-15 min to sanity-check actual steps/s before assuming it finishes in one
sitting. Cross-session resume (below) is there as a fallback if it runs long.

**Colab session budget: ~3h/day.** Checkpoints are written to Google Drive; re-run this
notebook on a later day (or a different account) to resume from the last checkpoint.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**Reclaim Drive quota (optional, run once per session):** the training script no
longer rotates/deletes checkpoints directly on Drive at all -- Trainer checkpoints on
local Colab disk and only ever *overwrites* one fixed Drive folder
(`{output_dir}/latest_checkpoint`), which sidesteps Drive's Trash-on-delete behavior
entirely. Still useful once, to clear out anything trashed by earlier runs. First run
prompts an auth popup.

**Warning:** this empties Trash for your **entire** Google Drive account, not just
this project's files -- anything else you'd trashed elsewhere and might still want
to recover will be gone permanently too. Skip this cell if that matters to you.

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
build("drive", "v3").files().emptyTrash().execute()
print("Drive Trash emptied.")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

`data/v2_ord_train/` is gitignored (large, derived) -- regenerate it deterministically
here (fixed seed, excludes the committed `data/v2_ord_eval_targets.json` by
construction). Only needs to run once per Colab session.

In [ ]:
import os

if not os.path.exists("data/v2_ord_train/conditions_train.jsonl"):
    !python scripts/build_train_data_ord.py --pool-count 60000 --seed 42

**Cross-account resume:** Colab sessions may run under different Google accounts/Drives
each time, so a previous session's `{output_dir}/latest_checkpoint` isn't reliably
visible automatically. Two ways to hand a checkpoint to the next session:

- **Recommended (faster for large checkpoints):** on drive.google.com (in whichever
  account is mounted *this* session), drag-and-drop the checkpoint folder anywhere
  under My Drive -- the Drive website handles large-folder uploads more reliably than
  a browser file picker. Then type its path into `resume_from_checkpoint_path` in the
  next cell -- skip the upload-widget cell entirely.
- **Alternative:** the upload-widget cell below (goes through the browser, slower for
  large folders).

Skip both for a first run.

In [ ]:
resume_from_checkpoint_path = ""  # @param {type:"string"}
# e.g. /content/drive/MyDrive/retro-planner-checkpoints/model2_conditions/final
# Leave blank if you're using the upload-widget cell below instead, or if this is a first run.

In [ ]:
import os
import shutil
import zipfile

from google.colab import files

uploaded = files.upload()  # skip this cell (don't run it) if you set resume_from_checkpoint_path above instead
if uploaded:
    zip_name = next(iter(uploaded))
    extract_dir = "/content/resume_from"
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(extract_dir)
    entries = os.listdir(extract_dir)
    if len(entries) == 1 and os.path.isdir(os.path.join(extract_dir, entries[0])):
        extract_dir = os.path.join(extract_dir, entries[0])
    resume_from_checkpoint_path = extract_dir
    print(f"Will resume from: {resume_from_checkpoint_path}")
    print("Contents:", os.listdir(resume_from_checkpoint_path))

In [ ]:
output_dir = "/content/drive/MyDrive/retro-planner-checkpoints/model2_conditions_reactiont5base"  # @param {type:"string"}
time_budget_minutes = 170  # @param {type:"number"}
base_model = "sagawa/ReactionT5v2-retrosynthesis"  # @param {type:"string"}

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"
resume_flag = ["--resume-from-checkpoint", resume_from_checkpoint_path] if resume_from_checkpoint_path else []

!python scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --learning-rate 5e-5 \
    --output-dir "{output_dir}" \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(resume_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Training output is redirected to `train.log` in `output_dir` (on Drive) instead of
printing here, to avoid the notebook's output growing large enough to make the browser
tab unresponsive on a long run. Open `train.log` in Google Drive's own web preview to
check progress -- that works independently of the Colab kernel, which stays busy
(blocked) running the cell above.

**To continue in a later session** (possibly under a different Google account): download
`{output_dir}/final` or a `checkpoint-N` from `{output_dir}/latest_checkpoint` as a zip,
then upload it in the "Cross-account resume" cell above next time. Same-account
reconnects auto-resume from `{output_dir}/latest_checkpoint` without needing an upload.
Once finished, evaluate against `data/v2_ord_train/conditions_test.jsonl` (held out, never
used in training) -- same file as `06_train_conditions_model.ipynb`'s t5-small run, so the
two are directly comparable.